In [17]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 00_data_quality_overview.py
# Purpose of Script: Provide Summary Statistics of Data Quality Checks.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd
import gc
import duckdb

In [19]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Print Versions
#~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f"Numpy version = {np.__version__}")
print(f"Pandas version = {pd.__version__}")
print(f"DuckDB version = {duckdb.__version__}")

Numpy version = 2.0.2
Pandas version = 2.2.2
DuckDB version = 1.3.2


In [20]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initiate Duck Connection
#~~~~~~~~~~~~~~~~~~~~~~~~~~
con = duckdb.connect()

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Input
path_samp = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/00_backups/01_sample_backups/02_20260629_final/"
path_data = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/00_backups/00_data_backups/02_data_backup_20260630/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

In [21]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Data Quality Analysis - Raw SOR DQ ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Data Quality Results Per File
df = con.execute(f""" select * from '{path_samp}dq_final.parquet'""").df()

In [23]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Raw DQ - Summary Statistics
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Summary - Per Platform
df_summary = (
    df
    .groupby("platform")
    .agg(files=("file", "count"),
        total_rows=("number_of_rows", "sum"),
        earliest_content=("minimum_content_date", "min"),
        latest_content=("maximum_content_date", "max"),
        earliest_sor=("minimum_sor_date", "min"),
        latest_sor=("maximum_sor_date", "max"),
        earliest_application=("minimum_application_date", "min"),
        latest_application=("maximum_application_date", "max"),
        missing_content_dates=("number_content_date_na", "sum"),
        missing_sor_dates=("number_sor_date_na", "sum"),
        missing_application_dates=("number_application_date_na", "sum"),
        territory_failures=("check_terr", lambda x: (x == "Y").sum()),
        uuid_failures=("check_uuids", lambda x: (x == "Y").sum()),
        platform_uuid_failures=("check_plat_uuids", lambda x: (x == "Y").sum()),).reset_index())

df_summary["platform"] = df_summary["platform"].str.capitalize()
df_summary = df_summary.sort_values(by = "total_rows", ascending=False).reset_index(drop=True)

# Add Overview Row
overall = pd.DataFrame({"platform": ["All Platforms"],
                        "files": [len(df)],
                        "total_rows": [df["number_of_rows"].sum()],
                        "earliest_content": [df["minimum_content_date"].min()],
                        "latest_content": [df["maximum_content_date"].max()],
                        "earliest_sor": [df["minimum_sor_date"].min()],
                        "latest_sor": [df["maximum_sor_date"].max()],
                        "earliest_application": [df["minimum_application_date"].min()],
                        "latest_application": [df["maximum_application_date"].max()],
                        "missing_content_dates": [df["number_content_date_na"].sum()],
                        "missing_sor_dates": [df["number_sor_date_na"].sum()],
                        "missing_application_dates": [df["number_application_date_na"].sum()],
                        "territory_failures": [(df["check_terr"] == "Y").sum()],
                        "uuid_failures": [(df["check_uuids"] == "Y").sum()],
                        "platform_uuid_failures": [(df["check_plat_uuids"] == "Y").sum()]})

df_summary = pd.concat([df_summary, overall], ignore_index=True)

In [25]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Outputs
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Overview Table
df_overview = df_summary[['platform','files','total_rows','earliest_content',
                          'latest_content','earliest_sor','latest_sor',
                          'earliest_application','latest_application']]

# Missing and Failures
df_miss_fail = df_summary[['platform','files','total_rows','missing_content_dates',
                           'missing_sor_dates','missing_application_dates',
                           'territory_failures','uuid_failures',
                           'platform_uuid_failures']]

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [26]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Missing/Failures
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# UUID Failures
df_uuid_facebook = df[(df['platform'] == 'facebook') & (df['check_uuids'] == "Y")]
df_uuid_instagram = df[(df['platform'] == 'instagram') & (df['check_uuids'] == "Y")]

# Platform UUID Failures
df_plat_uuid_facebook = df[(df['platform'] == 'facebook') & (df['check_plat_uuids'] == "Y")]
df_plat_uuid_instagram = df[(df['platform'] == 'instagram') & (df['check_plat_uuids'] == "Y")]
df_plat_uuid_tiktok = df[(df['platform'] == 'tiktok') & (df['check_plat_uuids'] == "Y")]
df_plat_uuid_whatsapp = df[(df['platform'] == 'whatsapp') & (df['check_plat_uuids'] == "Y")]

In [28]:
df_miss_fail

,platform,files,total_rows,missing_content_dates,missing_sor_dates,missing_application_dates,territory_failures,uuid_failures,platform_uuid_failures
0,Facebook,34022,631813529,0,0,0,0,1,12
1,Instagram,34604,165733954,0,0,0,0,1,8
2,Youtube,4028,140938874,0,0,0,0,0,0
3,Tiktok,6377,42543141,0,0,0,0,0,295
4,Snapchat,151544,6745969,0,0,0,0,0,0
5,X,17306,1759492,0,0,0,0,0,0
6,Whatsapp,528,334918,0,0,0,0,0,2
7,All Platforms,248409,989869877,0,0,0,0,2,317


In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Data Quality Analysis - Post Cleaning DQ ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_clean = con.execute(f""" select * from '{path_samp}dq_cleaning_final.parquet'""").df()

In [ ]:
# Sum all NAs
df_na_summary = (df_clean.groupby("platform").sum(numeric_only=True))
df_na_summary = df_na_summary.loc[:, df_na_summary.sum(axis=0) > 0]

In [ ]:
# All Overall Row
df_na_overall = pd.DataFrame(df_na_summary.drop(columns="platform").sum()).T
df_na_overall["platform"] = "All Platforms"

# Bind
df_na_summary = pd.concat([df_na_summary, df_na_overall], ignore_index=True)

KeyError: "['platform'] not found in axis"

In [ ]:
df_na_summary

,p_name,cont_lang,cat_spec_other,illegal_c_ground,illegal_c_ex,incomp_c_ground,incomp_c_ex,incomp_c_illegal,des_vis,des_vis_other,des_vis_end_date,des_mon,des_mon_other,des_mon_end_date,des_prov,des_prov_end_date,des_acc,des_acc_end_date,content_id_ean
platform,,,,,,,,,,,,,,,,,,,
facebook,0,631813529,631813529,310169248,310169248,0,0,631813529,503037700,631813529,631813529,630014971,631813529,631813529,631813529,631813529,130574387,631813529,369463294.0
instagram,0,165733954,165733954,122513253,122513253,0,0,165733954,140548427,165733954,165733954,165726719,165733954,165733954,165733954,165733954,25192762,165733954,92756289.0
snapchat,0,6745969,2012100,6714878,6714878,1,1,380289,1081837,6745969,6745969,6745969,6745969,6745969,6745969,6745969,5664132,6745969,4750554.0
tiktok,0,42543141,42543141,15493922,15493922,100,99,42543141,0,6,42288137,42543141,42543141,42507649,42122393,42112796,41846058,42533535,42543141.0
whatsapp,0,334918,334918,334918,334918,0,0,334918,0,334918,334918,334918,334918,334918,334918,334918,334918,334918,271818.0
x,1469,1759492,1319299,0,0,1759492,1759492,1759492,0,0,1759492,1759492,1759492,1759492,321783,1759492,685386,1759492,1307355.0
youtube,0,140938874,137877851,79200130,79200130,0,0,140938874,982551,140938874,140938874,140352908,140938874,140938874,139241048,140938874,140938874,140938874,88155490.0


In [ ]:
df_clean.head(10)

,platform,date,p_name,content_d,app_d,aut_det,aut_dec,cont_type,cont_lang,source,...,terr_mt,terr_nl,terr_no,terr_pl,terr_pt,terr_ro,terr_se,terr_si,terr_sk,version
0,facebook,2025-01-01,0,0,0,0,0,0,2843732,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
1,instagram,2025-01-01,0,0,0,0,0,0,292234,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
2,snapchat,2025-01-01,0,0,0,0,0,0,11485,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
3,whatsapp,2025-01-01,0,0,0,0,0,0,108,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
4,x,2025-01-01,0,0,0,0,0,0,1843,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
5,youtube,2025-01-01,0,0,0,0,0,0,135824,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
6,facebook,2025-01-02,0,0,0,0,0,0,3016407,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
7,instagram,2025-01-02,0,0,0,0,0,0,288786,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
8,snapchat,2025-01-02,0,0,0,0,0,0,11979,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final
9,whatsapp,2025-01-02,0,0,0,0,0,0,167,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20260629final


In [ ]:
df_clean.columns

Index(['platform', 'date', 'p_name', 'content_d', 'app_d', 'aut_det',
       'aut_dec', 'cont_type', 'cont_lang', 'source', 'cat', 'cat_spec',
       'cat_spec_other', 'des_ground', 'des_fact', 'illegal_c_ground',
       'illegal_c_ex', 'incomp_c_ground', 'incomp_c_ex', 'incomp_c_illegal',
       'des_vis', 'des_vis_other', 'des_vis_end_date', 'des_mon',
       'des_mon_other', 'des_mon_end_date', 'des_prov', 'des_prov_end_date',
       'des_acc', 'des_acc_end_date', 'terr', 'terr_eu_inc_eea',
       'terr_eu_ex_eea', 'content_id_ean', 'time', 'terr_at', 'terr_be',
       'terr_bg', 'terr_cy', 'terr_cz', 'terr_de', 'terr_dk', 'terr_ee',
       'terr_es', 'terr_fi', 'terr_fr', 'terr_gr', 'terr_hr', 'terr_hu',
       'terr_ie', 'terr_is', 'terr_it', 'terr_li', 'terr_lt', 'terr_lu',
       'terr_lv', 'terr_mt', 'terr_nl', 'terr_no', 'terr_pl', 'terr_pt',
       'terr_ro', 'terr_se', 'terr_si', 'terr_sk', 'version'],
      dtype='object')